# 跨会话记忆

> *在独立会话之间保存 Agent 状态，让回归用户获得连续性，而非一张白纸。*

想象一位每天失忆的同事。每天早上，你都得重新解释项目、偏好和昨天的决定。没有跨会话记忆的 AI Agent 就是这种感觉。最有价值的助手是那些能够学习和记住的。

没有跨会话记忆，每次对话都从零开始。一个花了三十分钟教 Agent 的用户第二天回来看见的是一张白纸。这不仅浪费用户时间，更限制了 Agent 能提供的价值。

跨会话记忆通过在会话之间添加持久化层来解决这个问题。"持久化"意味着保存数据，使程序停止运行后数据仍然存在。当会话结束时，Agent 保存其相关状态——包括提取的事实、用户偏好、对话摘要和任务上下文。系统将这个状态序列化（转换为可存储的格式）并写入持久化后端。当新会话开始时，系统识别回归用户，加载其记忆快照，并在第一轮对话前用这些上下文初始化 Agent。结果是一个能"记住"数天、数周甚至数月前内容的 Agent。

工程上的挑战实际但重要。你需要选择序列化格式：JSON 可读性强，Pickle 能处理复杂 Python 对象。你需要选择存储后端：Redis 速度快，SQLite 配置简单，S3 扩展性好。你还需要处理没有先前记忆时的冷启动情况，以及当已存储历史超过上下文窗口时决定加载多少先前上下文。

**本 notebook 中你将构建：**
1. 一个用于可序列化 Agent 状态的 `SessionState` 数据类。
2. 一个带 SQLite 实现的 `StorageBackend`。
3. 一个用于会话生命周期管理的 `CrossSessionManager`（保存、加载、冷启动）。
4. 一个在对话循环中使用持久化记忆的 `CrossSessionAgent`。
5. 同一用户跨多个会话的完整演示。

## 核心概念

- **会话序列化 (Session serialization)**：将 Agent 的内存状态转换为可存储的格式。状态包括对话历史、提取的事实、用户偏好和任务上下文。JSON 可读且易于检查。Pickle 能处理任意 Python 对象。Protobuf 提供紧凑的二进制编码并支持模式演化（在不破坏旧数据的前提下修改数据格式）。

- **状态持久化后端 (State persistence backends)**：会话之间序列化状态存放的持久化存储。Redis 提供亚毫秒级读取，适合低延迟恢复。SQLite 提供零依赖的本地持久化。S3/GCS 可扩展到数百万用户并具有高持久性。PostgreSQL 支持对存储状态的复杂查询。

- **会话恢复 (Session resumption)**：检测回归用户并加载其存储的记忆快照。系统在第一轮对话前还原（恢复）Agent 的内部状态。这必须足够快以避免会话开始时出现可感知的延迟。

- **用户识别 (User identification)**：将传入请求映射到持久的用户身份（用户 ID、API key、会话 token 或认证声明）。Agent 必须检索到正确的记忆分区。严格隔离防止记忆在不同用户之间泄漏。

- **记忆加载策略 (Memory loading strategies)**：当已存储历史超过上下文窗口时，系统必须选择加载什么。选项包括：完整历史、最近 N 轮、仅摘要、或相关性排序检索。相关性排序指在已存储记忆中查询与当前对话最相关的内容。

- **冷启动处理 (Cold start handling)**：当用户没有先前会话时初始化 Agent 状态，包括设置默认偏好和建立基线上下文。

## 模型准备

In [1]:
# 导入Langchain的初始化模型的函数
from langchain.chat_models import init_chat_model
# 加载环境变量
from dotenv import load_dotenv
load_dotenv()

# 调用init_chat_model函数初始化模型，参数model用来指定模型名称，Langchain会根据模型名字自动设定base_url，并从环境变量中获取api_key
model = init_chat_model(model="deepseek-chat")
print(type(model)) # <class 'langchain_deepseek.chat_models.ChatDeepSeek'>

/home/fish/ai-agent-notes/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


<class 'langchain_deepseek.chat_models.ChatDeepSeek'>


## 实现

我们将分四个部分构建跨会话记忆：

1. 一个 `SessionState` 数据类，持有 Agent 需要记住的一切。
2. 一个 `StorageBackend`，将状态持久化到 SQLite。
3. 一个 `CrossSessionManager`，处理保存、加载和冷启动。
4. 一个 `CrossSessionAgent`，将记忆接入对话循环。

### 会话状态

第一步是定义*要保存什么*。我们捕获五样东西：对话摘要、提取的事实、用户偏好、最近消息和会话元数据。

In [4]:
from dataclasses import dataclass, asdict

@dataclass
class SessionState:
    """会话之间持久化的 Agent 状态快照。"""

    user_id: str
    conversation_summary: str
    extracted_facts: list[str]
    user_preferences: dict
    last_n_messages: list[dict]
    session_count: int = 0
    last_active: str = ""

    def to_dict(self) -> dict:
        """转换为 JSON 可序列化的字典。"""
        return asdict(self)

    @classmethod
    def from_dict(cls, data: dict) -> "SessionState":
        """从字典重建 SessionState。"""
        return cls(**data)

### 存储后端

将存储后端想象成一个文件柜。每个抽屉标有用户 ID。每个抽屉里放着包含该用户会话状态的文件夹。后端的职责是归档和取出这些文件夹。

我们定义一个抽象接口，让你可以更换后端而不改动其余代码。然后我们构建一个基于 SQLite 的具体后端——一个只存在于单个文件中的轻量数据库。

In [ ]:
import os
import json
import sqlite3
from datetime import datetime
from abc import ABC, abstractmethod
from typing import Optional

class StorageBackend(ABC):
    """会话状态持久化接口。"""

    @abstractmethod
    def save(self, user_id: str, state: SessionState) -> None: ...

    @abstractmethod
    def load(self, user_id: str) -> Optional[SessionState]: ...

    @abstractmethod
    def delete(self, user_id: str) -> None: ...

    @abstractmethod
    def list_users(self) -> list[str]: ...


class SQLiteBackend(StorageBackend):
    """在 SQLite 数据库中持久化会话状态。"""

    def __init__(self, db_path: str = ":memory:"):
        self.conn = sqlite3.connect(db_path)
        self.conn.execute("""
            CREATE TABLE IF NOT EXISTS session_state (
                user_id    TEXT PRIMARY KEY,
                state_json TEXT NOT NULL,
                updated_at TEXT NOT NULL
            )
        """)
        self.conn.commit()

    def save(self, user_id: str, state: SessionState) -> None:
        self.conn.execute(
            """INSERT OR REPLACE INTO session_state
               (user_id, state_json, updated_at) VALUES (?, ?, ?)""",
            (user_id, json.dumps(state.to_dict()), datetime.now().isoformat()),
        )
        self.conn.commit()

    def load(self, user_id: str) -> Optional[SessionState]:
        row = self.conn.execute(
            "SELECT state_json FROM session_state WHERE user_id = ?",
            (user_id,),
        ).fetchone()
        if row is None:
            return None
        return SessionState.from_dict(json.loads(row[0]))

    def delete(self, user_id: str) -> None:
        self.conn.execute(
            "DELETE FROM session_state WHERE user_id = ?", (user_id,),
        )
        self.conn.commit()

    def list_users(self) -> list[str]:
        rows = self.conn.execute("SELECT user_id FROM session_state").fetchall()
        return [r[0] for r in rows]

### LLM 驱动的事实提取与摘要生成

在每个会话结束时，我们让 LLM 提取关于用户的关键事实，并将对话压缩为简短摘要。这些紧凑的表达将最重要的信息带入下一个会话，无需重放每条消息。

In [ ]:
from langchain_core.messages import SystemMessage, HumanMessage

def extract_facts(messages: list[dict], llm) -> list[str]:
    """让 LLM 从对话消息中提取用户关键事实。"""
    transcript = "\n".join(
        f"{m['role'].upper()}: {m['content']}" for m in messages
    )
    response = llm.invoke(
        [
            SystemMessage(content=(
                "从本对话中提取关于用户的关键事实。"
                "每条事实单独一行，以 '- ' 开头。"
                "重点关注：姓名、角色、项目、偏好和目标。"
            )),
            HumanMessage(content=transcript),
        ],
        max_tokens=300,
    )
    raw = response.content
    return [
        line.strip("- ").strip()
        for line in raw.strip().split("\n")
        if line.strip()
    ]


def summarize_conversation(messages: list[dict], llm) -> str:
    """将对话历史压缩为简短摘要。"""
    transcript = "\n".join(
        f"{m['role'].upper()}: {m['content']}" for m in messages
    )
    response = llm.invoke(
        [
            SystemMessage(content=(
                "用 2-3 句话总结本对话。"
                "捕获主要话题和任何做出的决定。"
            )),
            HumanMessage(content=transcript),
        ],
        max_tokens=200,
    )
    return response.content.strip()

### 跨会话管理器

管理器位于 Agent 和存储后端之间，处理三项职责：

1. **恢复**会话：加载已存储状态，递增会话计数器，应用加载策略。
2. **结束**会话：序列化当前状态并写入存储。
3. **冷启动**：当用户没有历史记录时创建默认状态。

In [ ]:
class CrossSessionManager:
    """编排会话持久化、恢复和冷启动。"""

    def __init__(
        self,
        backend: StorageBackend,
        loading_strategy: str = "full",
        max_messages: int = 20,
    ):
        self.backend = backend
        self.loading_strategy = loading_strategy
        self.max_messages = max_messages  # 用于 "last_n" 策略

    def resume_session(self, user_id: str) -> SessionState:
        """加载先前状态或为新用户冷启动。"""
        state = self.backend.load(user_id)
        if state is None:
            print(f"[冷启动] 用户 '{user_id}' 无先前状态。创建全新状态。")
            return self._cold_start(user_id)

        state.session_count += 1
        state.last_active = datetime.now().isoformat()
        loaded = self._apply_loading_strategy(state)
        print(
            f"[已恢复] 用户 '{user_id}'，会话 #{loaded.session_count}。"
            f"已加载 {len(loaded.last_n_messages)} 条消息，"
            f"{len(loaded.extracted_facts)} 条事实。"
        )
        return loaded

    def end_session(self, user_id: str, state: SessionState) -> None:
        """将当前状态持久化到后端。"""
        state.last_active = datetime.now().isoformat()
        self.backend.save(user_id, state)
        print(
            f"[已保存] 用户 '{user_id}'："
            f"{len(state.last_n_messages)} 条消息，"
            f"{len(state.extracted_facts)} 条事实已持久化。"
        )

接下来在 `CrossSessionManager` 上定义两个辅助方法。
`_cold_start` 为首次用户创建默认状态。
`_apply_loading_strategy` 根据所选策略（完整、最近 N 条或仅摘要）裁剪加载的状态。

In [ ]:
    def _cold_start(self, user_id: str) -> SessionState:
        """为首次用户初始化默认状态。"""
        return SessionState(
            user_id=user_id,
            conversation_summary="",
            extracted_facts=[],
            user_preferences={},
            last_n_messages=[],
            session_count=1,
            last_active=datetime.now().isoformat(),
        )

    def _apply_loading_strategy(self, state: SessionState) -> SessionState:
        """根据所选策略裁剪加载的状态。"""
        if self.loading_strategy == "full":
            return state
        elif self.loading_strategy == "last_n":
            # 仅保留最近的消息
            state.last_n_messages = state.last_n_messages[-self.max_messages :]
            return state
        elif self.loading_strategy == "summary":
            # 完全丢弃消息；仅依赖摘要 + 事实
            state.last_n_messages = []
            return state
        return state

### 组装：Agent

Agent 将跨会话记忆接入 LLM 对话循环。每轮它从已存储的摘要、提取事实和会话元数据中构建系统提示词。这让模型在读取新消息之前就拥有关于用户的上下文。

In [ ]:
class CrossSessionAgent:
    """具有跨会话持久记忆的对话 Agent。"""

    def __init__(self, manager: CrossSessionManager, llm=None):
        self.manager = manager
        self.llm = llm or model
        self.state: Optional[SessionState] = None
        self.user_id: Optional[str] = None

    def start_session(self, user_id: str) -> None:
        """为给定用户恢复或冷启动一个会话。"""
        self.user_id = user_id
        self.state = self.manager.resume_session(user_id)

    def chat(self, user_input: str) -> str:
        """发送消息并获取带跨会话上下文的回复。"""
        from langchain_core.messages import SystemMessage, HumanMessage, AIMessage

        self.state.last_n_messages.append(
            {"role": "user", "content": user_input}
        )

        # 从持久化记忆中构建系统提示词
        system_parts = [
            "你是一个具有跨会话记忆的有用助手。",
            "保持回复简洁（2-3 句话）。",
        ]
        if self.state.conversation_summary:
            system_parts.append(
                f"\n之前的对话摘要：\n{self.state.conversation_summary}"
            )
        if self.state.extracted_facts:
            facts_str = "\n".join(f"- {f}" for f in self.state.extracted_facts)
            system_parts.append(f"\n关于此用户的已知事实：\n{facts_str}")
        if self.state.session_count > 1:
            system_parts.append(
                f"\n这是与该用户的第 {self.state.session_count} 次会话。"
            )

        # 将 dict 消息转为 LangChain 消息对象
        lc_messages = [SystemMessage(content="\n".join(system_parts))]
        for m in self.state.last_n_messages:
            if m["role"] == "user":
                lc_messages.append(HumanMessage(content=m["content"]))
            elif m["role"] == "assistant":
                lc_messages.append(AIMessage(content=m["content"]))

        response = self.llm.invoke(lc_messages)
        reply = response.content

        self.state.last_n_messages.append(
            {"role": "assistant", "content": reply}
        )
        return reply

当会话结束时，Agent 提取事实并摘要对话，然后通过管理器持久化状态。这是实时对话与持久化存储之间的桥梁。

In [ ]:
    def end_session(self) -> None:
        """提取事实、摘要并持久化状态。"""
        if self.state and len(self.state.last_n_messages) > 0:
            self.state.extracted_facts = extract_facts(
                self.state.last_n_messages, self.llm
            )
            self.state.conversation_summary = summarize_conversation(
                self.state.last_n_messages, self.llm
            )
        self.manager.end_session(self.user_id, self.state)

## 示例运行

我们将模拟同一用户的两次会话，然后是一次新用户的冷启动。展示跨会话记忆如何创造连续性。

### 会话 1：初次见面

Alice 第一次与 Agent 对话。管理器因为无先前状态而触发冷启动。结束时我们保存她的会话。

In [ ]:
backend = SQLiteBackend()  # 本次演示使用内存 SQLite
manager = CrossSessionManager(backend, loading_strategy="full")
agent = CrossSessionAgent(manager)

agent.start_session("alice_123")

print("=" * 50)
print("会话 1：初次见面")
print("=" * 50, "\n")

session_1_messages = [
    "你好！我是 Alice，在 Acme Corp 担任数据科学家。",
    "我正在构建一个基于协同过滤的推荐系统。",
    "我偏好 Python，大部分数据处理工作都用 pandas。",
]

for msg in session_1_messages:
    print(f"用户:  {msg}")
    reply = agent.chat(msg)
    print(f"Agent: {reply}\n")

agent.end_session()

### 会话 2：Agent 记住了

Alice 稍后回来。Agent 加载她已存储的摘要和事实。它应该能在她不必重复的情况下回忆起她的名字、角色和项目。

In [ ]:
agent_s2 = CrossSessionAgent(manager)
agent_s2.start_session("alice_123")

print("=" * 50)
print("会话 2：Alice 回来了")
print("=" * 50, "\n")

session_2_messages = [
    "嘿，我回来了！你记得关于我的什么？",
    "对我的推荐系统有什么改进建议吗？",
]

for msg in session_2_messages:
    print(f"用户:  {msg}")
    reply = agent_s2.chat(msg)
    print(f"Agent: {reply}\n")

agent_s2.end_session()

### 冷启动：新用户

Bob 来了，没有历史记录。管理器创建默认状态，Agent 全新开始。

In [ ]:
agent_new = CrossSessionAgent(manager)
agent_new.start_session("bob_456")

print("=" * 50)
print("新用户：冷启动")
print("=" * 50, "\n")

msg = "你好！能帮我处理一个机器学习项目吗？"
print(f"用户:  {msg}")
reply = agent_new.chat(msg)
print(f"Agent: {reply}\n")

agent_new.end_session()

### 检查已存储状态

看看后端为 Alice 在两次会话后保存了什么。你将看到提取的事实、对话摘要和会话元数据。

In [ ]:
alice_state = backend.load("alice_123")

print("alice_123 的已存储状态")
print("-" * 40)
print(f"会话次数: {alice_state.session_count}")
print(f"最近活跃:   {alice_state.last_active}")
print(f"\n提取的事实:")
for fact in alice_state.extracted_facts:
    print(f"  - {fact}")
print(f"\n对话摘要:")
print(f"  {alice_state.conversation_summary}")
print(f"\n已存储消息: {len(alice_state.last_n_messages)} 条")
print(f"后端中的所有用户: {backend.list_users()}")

### 对比加载策略

当已存储历史增长过大时，你需要选择加载什么。这里我们使用 Alice 的已保存状态对比三种策略：

- **full**：加载全部。适合历史较短的用户。
- **last_n**：仅保留最近 N 条消息。裁剪旧上下文。
- **summary**：丢弃全部消息。仅依赖摘要和提取的事实。

In [ ]:
print("加载策略对比 (alice_123)")
print("=" * 50, "\n")

for strategy in ["full", "last_n", "summary"]:
    test_manager = CrossSessionManager(
        backend, loading_strategy=strategy, max_messages=4
    )
    loaded = test_manager.resume_session("alice_123")
    print(f"策略: {strategy}")
    print(f"  已加载消息:  {len(loaded.last_n_messages)}")
    print(f"  可用事实:  {len(loaded.extracted_facts)}")
    has_summary = "是" if loaded.conversation_summary else "否"
    print(f"  有摘要:    {has_summary}")
    print()

清理内存数据库。

In [ ]:
backend.conn.close()

## 权衡

### 适用场景

- **个性化助手。** 经常回归的用户获得更好的体验。Agent 记住他们的名字、偏好和进行中的项目。
- **长期任务。** 一个持续多日的研究项目可以从中断处继续。Agent 追踪已覆盖的内容和剩余内容。
- **减少用户操作。** 用户无需重复上下文。这节省时间并减少挫败感。

### 失效场景

- **隐私与数据留存。** 跨会话存储用户数据引发合规问题（GDPR、CCPA）。你需要明确数据保存时长和用户删除政策。
- **记忆陈旧。** 数月前提取的事实可能不再真实。用户可能换了工作或切换了项目。没有过期旧事实的机制，Agent 可能做出错误假设。
- **大规模存储成本。** 数百万用户时，存储后端成为真正的基础设施问题。你需要规划数据库规模、备份和访问模式。
- **上下文窗口限制。** 即使有加载策略，多会话累积的状态也可能超出单次提示词容量。你需要将此技术与摘要或检索方案结合使用。

## 延伸阅读

- Packer, C., et al. (2023). ["MemGPT: Towards LLMs as Operating Systems."](https://arxiv.org/abs/2310.08560) 为 LLM 引入虚拟内存分页。Agent 在有界上下文窗口中工作，但从持久化外部存储中按需调页数据。

- [LangChain 记忆文档](https://python.langchain.com/docs/modules/memory/?utm_source=nirdiamant&utm_medium=github&utm_campaign=agent_memory_techniques)：涵盖对话记忆类型（缓冲、摘要、实体），支持文件和数据库持久化。

- [Letta (MemGPT) 持久化模型](https://docs.letta.com?utm_source=nirdiamant&utm_medium=github&utm_campaign=agent_memory_techniques)：记录了 Letta 跨会话持久化 Agent 状态的方法，包括记忆层次结构和后端架构。

- [SQLite：何时使用](https://www.sqlite.org/whentouse.html?utm_source=nirdiamant&utm_medium=github&utm_campaign=agent_memory_techniques)：关于 SQLite 何时是正确数据库选择的指南。许多 Agent 原型使用 SQLite 进行本地状态持久化。

- [Redis 持久化文档](https://redis.io/docs/latest/operate/oss_and_stack/management/persistence/)：解释 Redis 持久化选项（RDB 快照、AOF 日志）。当需要生产级亚毫秒会话恢复时很有用。

---

*← 上一章：[20：记忆检索模式](../20_memory_retrieval_patterns/) · 下一章：[22：多 Agent 共享记忆](../22_multi_agent_shared_memory/) →*

## 🧪 自己动手试试

三个小挑战来加深你的理解。每个应该花费 10-30 分钟。

### 挑战 1：状态版本管理
为 `SessionState` 添加一个 `version` 整数字段。每次 `CrossSessionManager` 保存状态时递增版本号，并在 `history` 表中保留上一版本。实现一个 `rollback(version)` 方法来恢复先前状态。通过保存 3 个版本并回滚到第一个版本来测试。

### 挑战 2：存储增长追踪
对话中每 5 轮后，将 `SessionState` 序列化为 JSON 并记录其字节大小。运行 30 轮对话，绘制轮次与状态大小的关系图。识别哪个组件（消息、事实或偏好）增长最快。

### 挑战 3：多用户会话管理器
扩展 `CrossSessionManager` 使其接受 `user_id` 参数。将每个用户的状态存储到不同的 SQLite 行（或文件）中。创建两个用户，各运行 5 轮，保存，然后恢复两者。验证其状态是隔离的。为 22 多 Agent 共享记忆中的多 Agent 模式做铺垫。

![](https://europe-west1-amt-views-tracker.cloudfunctions.net/amt-tracker?notebook=all-techniques--21-cross-session-memory--cross-session-memory)
